# שבוע 5: יישור פרוקרוסטס (GPA)

בשיעור זה נלמד:
- מה עושה ה-GPA (Generalized Procrustes Analysis)
- כיצד להריץ GPA עם ספריית morphops
- גודל-צנטרואיד כמדד גודל
- הבדל ויזואלי לפני ואחרי יישור

> **הוראות**: הריצו כל תא בסדר מלמעלה למטה. לחצו על התא ואחר כך `Shift+Enter`.

In [ ]:
!pip install morphops python-bidi -q
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from bidi.algorithm import get_display
rtl = get_display
print('הכל מוכן!')

## טעינת הנתונים

נשתמש בנתוני מטבעות הדריאנוס ואנטונינוס פיוס.

In [ ]:
import urllib.request

def parse_tps(text):
    specimens, ids = [], []
    lines = text.strip().split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('LM='):
            n_lm = int(line.split('=')[1])
            coords = []
            for j in range(n_lm):
                i += 1
                parts = lines[i].strip().replace(',', '.').split()
                coords.append([float(parts[0]), float(parts[1])])
            specimens.append(np.array(coords))
        elif line.startswith('ID='):
            ids.append(line.split('=')[1])
        i += 1
    return np.array(specimens), ids

def make_synthetic_coins(n, seed_offset=0):
    """נתוני מטבעות סינתטיים לגיבוי"""
    np.random.seed(42 + seed_offset)
    coins = []
    for i in range(n):
        cx = np.random.uniform(100, 500)
        cy = np.random.uniform(100, 500)
        scale = np.random.uniform(0.8, 1.2)
        angle_offset = np.random.uniform(0, 0.2)
        lm = []
        for j in range(8):
            a = 2*np.pi*j/8 + angle_offset
            r = 80*scale + np.random.randn()*3
            lm.append([cx + r*np.cos(a), cy + r*np.sin(a)])
        coins.append(np.array(lm))
    return np.array(coins), [f'coin_{i+1:03d}' for i in range(n)]

base = 'https://raw.githubusercontent.com/shaigordin/comparch/2026/morphometrics/data/coins/'
try:
    with urllib.request.urlopen(base + 'hadrian.tps') as r:
        lm_h, ids_h = parse_tps(r.read().decode('utf-8'))
    with urllib.request.urlopen(base + 'antoninus.tps') as r:
        lm_a, ids_a = parse_tps(r.read().decode('utf-8'))
    print(f'הדריאנוס: {len(lm_h)} מטבעות')
    print(f'אנטונינוס פיוס: {len(lm_a)} מטבעות')
except Exception as e:
    print(f'משתמשים בנתוני דוגמה: {e}')
    lm_h, ids_h = make_synthetic_coins(20, 0)
    lm_a, ids_a = make_synthetic_coins(15, 10)

all_lm = np.concatenate([lm_h, lm_a], axis=0)
labels = np.array(['הדריאנוס'] * len(lm_h) + ['אנטונינוס פיוס'] * len(lm_a))
print(f'\nסה"כ: {len(all_lm)} מטבעות')

## מה זה GPA?

GPA (Generalized Procrustes Analysis) מכוון את כל הפרטים על ידי:

1. **הזזה** — מרכז כל פרט לנקודת המקור (0,0)
2. **שינוי קנה-מידה** — מנרמל לגודל-צנטרואיד = 1
3. **סיבוב** — ממזער את סכום המרחקים הריבועיים בין ציוני הדרך המתאימים

**מה נשמר**: צורה בלבד — גדול, מיקום וסיבוב מוסרים.

In [ ]:
import morphops as mops

# הרצת GPA
aligned, mean_shape, _ = mops.procrustes(all_lm)
print(f'צורת מוצא: {all_lm.shape}')
print(f'לאחר GPA:  {aligned.shape}')
print(f'צורת ממוצע: {mean_shape.shape}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.set_title(rtl('לפני GPA (קואורדינטות גולמיות)'), fontsize=13)
for lm in all_lm[:10]:
    ax1.plot(lm[:, 0], lm[:, 1], 'o-', alpha=0.4, markersize=4)
ax1.set_aspect('equal')
ax1.invert_yaxis()
ax1.set_xlabel(rtl('X (פיקסלים)'))
ax1.set_ylabel(rtl('Y (פיקסלים)'))

ax2.set_title(rtl('אחרי GPA (קואורדינטות פרוקרוסטס)'), fontsize=13)
for lm in aligned[:10]:
    ax2.plot(lm[:, 0], lm[:, 1], 'o-', alpha=0.4, markersize=4)
ax2.plot(mean_shape[:, 0], mean_shape[:, 1], 'k*-', markersize=12,
         linewidth=2, label=rtl('צורת ממוצע'))
ax2.set_aspect('equal')
ax2.legend()
ax2.set_xlabel(rtl('X (יחידות פרוקרוסטס)'))

plt.tight_layout()
plt.show()

## גודל-צנטרואיד

גודל-צנטרואיד (CS) = שורש הריבועי של סכום המרחקים הריבועיים של ציוני הדרך מהמרכז.
הוא המדד הסטנדרטי לגודל בריפומטריה גאומטרית.

In [ ]:
def centroid_size(lm):
    centroid = lm.mean(axis=0)
    return np.sqrt(np.sum((lm - centroid)**2))

cs_h = [centroid_size(lm) for lm in lm_h]
cs_a = [centroid_size(lm) for lm in lm_a]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(cs_h, bins=8, alpha=0.7, label=rtl('הדריאנוס'), color='steelblue')
ax.hist(cs_a, bins=8, alpha=0.7, label=rtl('אנטונינוס פיוס'), color='coral')
ax.set_xlabel(rtl('גודל-צנטרואיד'))
ax.set_ylabel(rtl('מספר מטבעות'))
ax.set_title(rtl('השוואת גדלי מטבעות'))
ax.legend()
plt.tight_layout()
plt.show()

print(f'ממוצע CS הדריאנוס:    {np.mean(cs_h):.1f} ± {np.std(cs_h):.1f}')
print(f'ממוצע CS אנטונינוס:   {np.mean(cs_a):.1f} ± {np.std(cs_a):.1f}')

## תרגיל

1. מדוע נחלק ב-CS ולא פשוט בשטח או בהיקף?
2. מה קורה לשונות בגרף השני (לאחר GPA) לעומת הראשון (לפני)? מדוע?
3. האם גדל מטבעות הדריאנוס שונה מגדל מטבעות אנטונינוס פיוס? מה אתם רואים בהיסטוגרמה?